# W9-D2 概念实验：SkillRelease 为什么是唯一可部署单元？

配套阅读：同名 `.md`。这里不复述阅读材料，而是用小规模、可运行的模型检验其中的架构约束。

## 实验问题

**问题 1：为什么 Blueprint、IR 和 Capability 都不该直接成为部署入口？**

用类型和门禁模拟：源蓝图不可执行，IR 是内部表示，Capability 是依赖；只有通过治理门的 release 可部署。

In [ ]:
from dataclasses import dataclass, field, replace
from hashlib import sha256
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(202608)
def canonical_digest(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "sha256:" + sha256(canonical.encode()).hexdigest()
@dataclass(frozen=True)
class Blueprint: digest: str
@dataclass(frozen=True)
class ExecutionPlanIR: digest: str
@dataclass(frozen=True)
class Capability: digest: str
@dataclass(frozen=True)
class SkillRelease:
    digest: str
    ir_digest: str
    capability_digests: tuple
    evaluation_passed: bool
    approval_signed: bool

def deploy(unit):
    if not isinstance(unit, SkillRelease):
        raise TypeError(f"{type(unit).__name__} 不是部署边界")
    if not (unit.evaluation_passed and unit.approval_signed):
        raise ValueError("release gate 未通过")
    return f"deployment<-{unit.digest[:16]}"

bp, ir, cap = Blueprint("sha256:bp"), ExecutionPlanIR("sha256:ir"), Capability("sha256:cap")
for item in [bp, ir, cap]:
    try: deploy(item)
    except Exception as exc: print(type(exc).__name__, "-", exc)
release = SkillRelease("sha256:release-a", ir.digest, (cap.digest,), True, True)
print("允许：", deploy(release))

## 实验问题

**问题 2：打包与依赖锁到底避免了什么？**

构造两个看似同名的“合同查询”发布包。release digest 把 IR、能力、知识和策略的精确版本一起锁住。

In [ ]:
def release_digest(ir, capabilities, knowledge, policy):
    return canonical_digest({"ir": ir, "capabilities": sorted(capabilities), "knowledge": knowledge, "policy": policy})

r_a = release_digest("sha256:ir1", ["sha256:cap1"], "sha256:kb1", "sha256:policy1")
r_b = release_digest("sha256:ir1", ["sha256:cap1"], "sha256:kb2", "sha256:policy1")
print("知识快照 kb1 的 release:", r_a[:24] + "...")
print("知识快照 kb2 的 release:", r_b[:24] + "...")
assert r_a != r_b
print("同一 skill 名称不足以描述可执行内容；release digest 才是闭合身份。")

## 实验问题

**问题 3：唯一入口如何减少可见的攻击/误用面？**

枚举四类内部对象可以暴露的端点数量；将对外调用集中到 SkillRelease 后，客户端只需理解稳定 schema。

In [ ]:
surfaces = {"Blueprint": ["edit", "build"], "Capability": ["invoke"], "Workflow": ["start", "resume"], "SkillRelease": ["describe", "invoke", "status"]}
public = {"SkillRelease": surfaces["SkillRelease"]}
print("若全部暴露，客户端需依赖端点：", sum(map(len, surfaces.values())))
print("唯一发布单元暴露端点：", sum(map(len, public.values())))
print("公开契约：", public)
assert "invoke" in public["SkillRelease"] and len(public) == 1

## 实验问题

**问题 4：为什么 Release Gate 是治理边界，而不是部署脚本里的可选检查？**

生成 60 个候选包，随机赋予评估与签名状态，观察只有双重满足的包才可进入部署队列。

In [ ]:
n = 60
evaluated = rng.random(n) < .82
signed = rng.random(n) < .78
eligible = evaluated & signed
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.bar(["候选", "评估通过", "评估+签名\n可部署"], [n, evaluated.sum(), eligible.sum()], color=["#7570b3", "#1b9e77", "#66a61e"])
ax.set_ylabel("发布包数量"); ax.set_title("SkillRelease 将治理证据封装为部署资格")
plt.tight_layout(); plt.show()
print(f"可部署比例：{eligible.mean():.1%}；其余包仍可保存，但不能被 Runtime 消费。")